# Flagship CIC-IDS2017 model
I will use a day-by-day split to simulate real-world intrustion detection with zero-day (unseen) attacks. 

In [29]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import glob
import re
import unicodedata
from sklearn.model_selection import train_test_split

RANDOM_STATE = 5
import time

# Supervised Baseline

In [30]:
files = glob.glob("data/*.csv")

day_re = "/([a-zA-Z]*)-"

def read_day_csv(f):
    df = pd.read_csv(f, encoding_errors="replace")
    df["day"] = re.search(day_re, f).group(1)
    return df

df = (
    pd.concat((read_day_csv(f) for f in files), ignore_index=True)
    .rename(columns=lambda s: s.strip())
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

df = df.drop(columns=["Destination Port", "Init_Win_bytes_forward",
            "Fwd Header Length", "Total Fwd Packets", "min_seg_size_forward",
            "SYN Flag Count", "Bwd Packet Length Mean"])

# Safely downcasts 64-bit data to 32-bit where there is no information loss
def down_cast(d):
    original_size = d.memory_usage(deep=True).sum()
    for col in d.select_dtypes("integer").columns:
        d[col] = pd.to_numeric(d[col], downcast="integer")
    for col in d.select_dtypes("float").columns:
        d[col] = pd.to_numeric(d[col], downcast="float")
    final_size = d.memory_usage(deep=True).sum()
    return d
df = down_cast(df)

def normalise(s):
    return (unicodedata.normalize("NFKD", str(s))
            .encode("ascii", "ignore")
            .decode("ascii"))
df["Label"] = df["Label"].apply(normalise)

train_days = df["day"].isin(["Monday", "Tuesday", "Wednesday"])
train = df[train_days]
test  = df[~train_days]

train_attacks = set(train["Label"]) - {"BENIGN"}
test_attacks  = set(test["Label"])  - {"BENIGN"}
novel         = test_attacks - train_attacks

X_train = train.drop(columns=["Label", "day"])
X_test  = test.drop(columns=["Label", "day"])

y_train_attackname = train["Label"]
y_test_attackname  = test["Label"]

multi_to_binary = lambda name: 0 if (name=="BENIGN") else 1
y_train = y_train_attackname.map(multi_to_binary)
y_test = y_test_attackname.map(multi_to_binary)

X_train, X_cv, y_train, y_cv = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE)

# For XGBoost, don't need scaling
# from sklearn.preprocessing import StandardScaler
# std_scaler = StandardScaler().fit(X_train)
# X_train = std_scaler.transform(X_train)
# X_cv    = std_scaler.transform(X_cv)
# X_test  = std_scaler.transform(X_test)

In [9]:
from sklearn.metrics import classification_report

def evaluate(preds, y, target_names, title=""):
    print(f"--- {title} ---")
    print(f"accuracy: {(preds == y).mean():.4f}")
    print(classification_report(y, preds, target_names=target_names))
    
hyper_params = {"n_estimators": 300, "learning_rate": 0.1, "verbosity": 1,
             "random_state": RANDOM_STATE,  "early_stopping_rounds": 20, "tree_method": "hist"}
eval_set = [(X_cv, y_cv)]

xgb_model = XGBClassifier(**hyper_params)
xgb_model.fit(X_train, y_train, eval_set = eval_set, verbose=0)

preds = xgb_model.predict(X_test)
evaluate(preds, y_test, ["benign", "anomaly"], "baseline")

del xgb_model

--- baseline ---
accuracy: 0.7426
              precision    recall  f1-score   support

      benign       0.75      0.99      0.85    870343
     anomaly       0.18      0.01      0.02    291001

    accuracy                           0.74   1161344
   macro avg       0.47      0.50      0.43   1161344
weighted avg       0.61      0.74      0.64   1161344



In [10]:
results = pd.DataFrame({
    "attack": y_test_attackname.values,   # raw string labels you kept
    "pred":   preds,                      # 0/1 from the model
})
# recall per type = fraction of that type's rows flagged as attack (pred==1)
per_type = results[results["attack"] != "BENIGN"].groupby("attack")["pred"].agg(
    recall="mean", n="size"
)
print(per_type.sort_values("recall"))

                             recall       n
attack                                     
Bot                        0.000000    1956
Infiltration               0.000000      36
PortScan                   0.000579  158804
DDoS                       0.006671  128025
Web Attack  Sql Injection  0.333333      21
Web Attack  Brute Force    0.595222    1507
Web Attack  XSS            0.687117     652


# Isolation Forest
First we have to remove anomalous examples from the training set. Otherwise the iForest will learn anomalous behaviour as normal! We also no longer require a cross validation set, allowing a doubling in the test set.

In [60]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

In [61]:
files = glob.glob("data/*.csv")

day_re = "/([a-zA-Z]*)-"

def read_day_csv(f):
    df = pd.read_csv(f, encoding_errors="replace")
    df["day"] = re.search(day_re, f).group(1)
    return df

df = (
    pd.concat((read_day_csv(f) for f in files), ignore_index=True)
    .rename(columns=lambda s: s.strip())
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

df = df.drop(columns=["Destination Port", "Init_Win_bytes_forward",
            "Fwd Header Length", "Total Fwd Packets", "min_seg_size_forward",
            "SYN Flag Count", "Bwd Packet Length Mean"])

# Safely downcasts 64-bit data to 32-bit where there is no information loss
def down_cast(d):
    original_size = d.memory_usage(deep=True).sum()
    for col in d.select_dtypes("integer").columns:
        d[col] = pd.to_numeric(d[col], downcast="integer")
    for col in d.select_dtypes("float").columns:
        d[col] = pd.to_numeric(d[col], downcast="float")
    final_size = d.memory_usage(deep=True).sum()
    return d
df = down_cast(df)

def normalise(s):
    return (unicodedata.normalize("NFKD", str(s))
            .encode("ascii", "ignore")
            .decode("ascii"))
df["Label"] = df["Label"].apply(normalise)

In [62]:
def make_splits(df, train_days=("Monday", "Tuesday", "Wednesday"),
                cv_frac=0.2, random_state=0):
    # Splits train and test temporally by days
    # All training-day(s) attacks go to CV
    # Returns a dict of frames + novel-attack set
    is_train_day = df["day"].isin(list(train_days))
    train_all = df[is_train_day]
    test = df[~is_train_day]
 
    # compute novelty against the FULL training-day label set, pre-filter
    train_attacks = set(train_all["Label"]) - {"BENIGN"}
    test_attacks = set(test["Label"]) - {"BENIGN"}
    novel = test_attacks - train_attacks
 
    benign = train_all[train_all["Label"] == "BENIGN"]
    attacks = train_all[train_all["Label"] != "BENIGN"]
 
    fit, cv_benign = train_test_split(
        benign, test_size=cv_frac, random_state=random_state
    )
    cv = pd.concat([cv_benign, attacks], ignore_index=True)
 
    drop = ["Label", "day"]
    to_binary = lambda s: np.where(s == "BENIGN", 1, -1)
 
    return {
        "X_fit": fit.drop(columns=drop),
        "X_cv": cv.drop(columns=drop),
        "X_test": test.drop(columns=drop),
        "y_cv": to_binary(cv["Label"]),
        "y_test": to_binary(test["Label"]),
        "cv_names": cv["Label"].to_numpy(),
        "test_names": test["Label"].to_numpy(),
        "novel": novel,
        "seen": train_attacks,
    }

In [56]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

maxes = [128, 256, 1024, 4096, 16384]

for max_samples in maxes:
    model = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=RANDOM_STATE,
        max_samples=max_samples
    )
    model.fit(X_train)
    preds = model.predict(X_test)
    evaluate(preds, y_test, ["anomaly", "benign"], f"iForest {max_samples}")
    

# clf = IsolationForest(random_state=RANDOM_STATE).fit(X_train)
# preds = clf.predict(X_test)

# evaluate(preds, y_test, ["anomaly", "benign"], "iForest")

--- iForest 128 ---
accuracy: 0.7319
              precision    recall  f1-score   support

     anomaly       0.44      0.28      0.34    291001
      benign       0.79      0.88      0.83    870343

    accuracy                           0.73   1161344
   macro avg       0.62      0.58      0.59   1161344
weighted avg       0.70      0.73      0.71   1161344

--- iForest 256 ---
accuracy: 0.7529
              precision    recall  f1-score   support

     anomaly       0.51      0.28      0.36    291001
      benign       0.79      0.91      0.85    870343

    accuracy                           0.75   1161344
   macro avg       0.65      0.60      0.60   1161344
weighted avg       0.72      0.75      0.73   1161344



KeyboardInterrupt: 

In [55]:
maxes = [0.1, 0.2, 0.4, 0.5, 0.75, 1]

for max_features in maxes:
    model = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=RANDOM_STATE,
        max_features=max_features,
    )
    model.fit(X_train)
    preds = model.predict(X_test)
    evaluate(preds, y_test, ["anomaly", "benign"], f"iForest {max_features}")

--- iForest 0.1 ---
accuracy: 0.7078
              precision    recall  f1-score   support

     anomaly       0.25      0.08      0.12    291001
      benign       0.75      0.92      0.82    870343

    accuracy                           0.71   1161344
   macro avg       0.50      0.50      0.47   1161344
weighted avg       0.62      0.71      0.65   1161344

--- iForest 0.2 ---
accuracy: 0.7074
              precision    recall  f1-score   support

     anomaly       0.26      0.09      0.13    291001
      benign       0.75      0.91      0.82    870343

    accuracy                           0.71   1161344
   macro avg       0.50      0.50      0.48   1161344
weighted avg       0.63      0.71      0.65   1161344

--- iForest 0.4 ---
accuracy: 0.7072
              precision    recall  f1-score   support

     anomaly       0.26      0.09      0.14    291001
      benign       0.75      0.91      0.82    870343

    accuracy                           0.71   1161344
   macro avg     

In [ ]:
evaluate(pred, y_test, ["anomaly", "benign"], "iForest")

results = pd.DataFrame({
    "attack": y_test_attackname.values,   # raw string labels you kept
    "pred":   (preds == -1).astype(int),  # 1 if attack, else 0
})
# recall per type = fraction of that type's rows flagged as attack (pred==1)
per_type = results[results["attack"] != "BENIGN"].groupby("attack")["pred"].agg(
    recall="mean", n="size"
)
print(per_type.sort_values("recall"))